In [ ]:
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

# CDS Monthly

In [ ]:
t2m = xr.open_mfdataset("data/era5_monthly/era5_monthly_precipitation_*.nc", combine="by_coords", parallel=True)
t2m.close()
tp  = xr.open_mfdataset("data/era5_monthly/era5_monthly_temperature_*.nc",  combine="by_coords", parallel=True)
tp.close()
vpd = xr.open_mfdataset("data/era5_derived_monthly.nc", combine="by_coords", parallel=True)
vpd.close()


t2m = t2m.assign_coords(valid_time=t2m.valid_time.dt.floor("D"))
tp = tp.assign_coords(valid_time=tp.valid_time.dt.floor("D"))
vpd = vpd.assign_coords(valid_time=vpd.valid_time.dt.floor("D"))

ds_monthly = xr.merge([t2m, tp, vpd], compat="override")
ds_monthly = ds_monthly.drop_vars("number")
    
print(ds_monthly)

In [ ]:
# Apply land-sea mask:

lsm = xr.open_dataset("data/era5_static/era5_lsm.nc")["lsm"].squeeze()
lsm = lsm.reindex_like(ds_monthly, method="nearest")
ds_monthly_land = ds_monthly.where(lsm > 0.5)

In [ ]:
print(f'Number of valid times: {ds_monthly.sizes["valid_time"]}') # Should be 312 = 26 years x 12 months
print(f'Number of nulls in t2m: {ds_monthly["t2m"].isnull().sum().compute().item()}') # Should be zero
print(f'Number of nulls in tp: {ds_monthly["tp"].isnull().sum().compute().item()}') # Should be zero


In [ ]:
quantiles = [0, 0.25, 0.5, 0.75, 1.0]

q = (
    ds_monthly_land[["t2m", "tp", "vpd_monthly_mean_daymax", "wind_speed_monthly"]]
    .groupby("valid_time.month")
    .quantile(quantiles, dim=...)   # ... means "all remaining dims"
    .compute()
)

df = q.to_dataframe().unstack("quantile").round(2)
df.index = pd.to_datetime(df.index, format="%m").strftime("%b")
df.index.name = "month"

ds_monthly_display = ds_monthly_land.assign(
    t2m=ds_monthly_land["t2m"] - 273.15,
    tp=ds_monthly_land["tp"] * ds_monthly.valid_time.dt.days_in_month * 1000,  # mm/month

)
ds_monthly_display.t2m.attrs["long_name"] = "Temperature at 2 meters"
ds_monthly_display.t2m.attrs["units"] = "ºC"
ds_monthly_display.tp.attrs["long_name"] = "Total precipitation"
ds_monthly_display.tp.attrs["units"] = "mm/month"
ds_monthly_display.vpd_monthly_mean_daymax.attrs["long_name"] = "VPD Daily Maximum"
ds_monthly_display.vpd_monthly_mean_daymax.attrs["units"] = "hPA"
ds_monthly_display.wind_speed_monthly.attrs["long_name"] = "Wind speed"
ds_monthly_display.wind_speed_monthly.attrs["units"] = "m/s"

ds_monthly_display = ds_monthly_display.drop_vars("number")

q = (
    ds_monthly_display[["t2m", "tp", "vpd_monthly_mean_daymax", "wind_speed_monthly"]]
    .groupby("valid_time.month")
    .quantile(quantiles, dim=...)
    .compute()
)

df = q.to_dataframe().unstack("quantile").round(2)
df.index = pd.to_datetime(df.index, format="%m").strftime("%b")

print("Temperature at 2 meters in ºC")
print(df["t2m"])
print("-------------------------------------------------")
print("Total precipitation in mm/month")
print(df["tp"])
print("-------------------------------------------------")
print("Vapor Pressure Deficit (VPD) daily maximum in hPA")
print(df["vpd_monthly_mean_daymax"])
print("-------------------------------------------------")
print("Wind speed in m/s")
print(df["wind_speed_monthly"])


In [ ]:
ds_monthly_display.tp.mean(dim="valid_time").plot(figsize=(10, 6))
plt.title("Mean monthly precipitation, Iberia 2000–2025")
plt.show()


In [ ]:
ds_monthly_display.t2m.mean(dim="valid_time").plot(figsize=(10, 6))
plt.title("Mean monthly temperature at 2 meters, Iberia 2000–2025")
plt.show()

In [ ]:
ds_monthly_display.vpd_monthly_mean_daymax.mean(dim="valid_time").plot(figsize=(10, 6))
plt.title("Mean VPD daily maximum, Iberia 2000–2025")
plt.show()

In [ ]:
ds_monthly_display.wind_speed_monthly.mean(dim="valid_time").plot(figsize=(10, 6))
plt.title("Mean wind speed, Iberia 2000–2025")
plt.show()